# CS 3110/3990: Data Privacy

## In-Class Exercise #4, week of 9/28/2026


In [1]:
# Load the data and libraries
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt


def laplace_mech(v, sensitivity, epsilon):
    return v + np.random.laplace(loc=0, scale=sensitivity / epsilon)


def pct_error(orig, priv):
    return np.abs(orig - priv) / orig * 100.0


adult = pd.read_csv(
    "https://github.com/jnear/cs3110-data-privacy/raw/main/homework/adult_with_pii.csv"
)

## Question 1

For various values of $b$, write code to print out the percent error of summing the ages in the `adult` dataset,
clipped to each value of $b$.


In [2]:
bs = range(1, 100, 10)
real_sum = adult["Age"].sum()

for b in bs:
    clipped_sum = adult["Age"].clip(lower=0, upper=b).sum()
    print("Percent error for", b, "is", pct_error(real_sum, clipped_sum))

# We don't want to pick a value too low, we want to be as close to the maximum
# of the dataset as possible.

Percent error for 1 is 97.4080940444511
Percent error for 11 is 71.48903448896206
Percent error for 21 is 46.000380495392264
Percent error for 31 is 25.509031989473492
Percent error for 41 is 11.7959939725709
Percent error for 51 is 4.329687317165198
Percent error for 61 is 1.1587597123836921
Percent error for 71 is 0.2341877497996031
Percent error for 81 is 0.03860675005194001
Percent error for 91 is 0.0


What value of $b$ is the best?


Any value of b larger than 81 is probably fine, because at b=81 the relative
error due to bias is small (0.03%). Values of b much larger than this risk
adding additional error due to variance when we add noise to achieve differential privacy.


## Question 2

For various values of $b$, print the result of a _differentially private_ sum of ages, clipped to each value of $b$. Use $\epsilon = 0.1$.


In [3]:
bs = range(1, 200, 20)
real_sum = adult["Age"].sum()
print("number of candidates:", len(bs))

# This program satisfies differential privacy for a total privacy cost of
# epsilon = 10 * 0.1 by sequential composition (len(bs) * epsilon)
# epsilon=0.1 per query.
for b in bs:
    clipped_sum = adult["Age"].clip(lower=0, upper=b).sum()
    dp_sum = laplace_mech(clipped_sum, sensitivity=b, epsilon=0.1)
    print(
        "DP result for",
        b,
        "is",
        dp_sum,
        # "| error is",
        # pct_error(real_sum, dp_sum),
    )

# We want to minimize the total error, so by looking at various values of b and
# calculating the corresponding errors, we can choose the one that gives us the
# smallest error. however, looking at the percent error violates differential privacy.

# The error here now has some randomness due to the Laplace mechanism.

# One thing we could do is look for the first time when the value goes down./

number of candidates: 10
DP result for 1 is 32559.896772696513
DP result for 21 is 678467.9851651983
DP result for 41 is 1108300.8478421578
DP result for 61 is 1243113.7568131597
DP result for 81 is 1255379.969441192
DP result for 101 is 1256470.1856329192
DP result for 121 is 1255612.4252797118
DP result for 141 is 1256766.3651824866
DP result for 161 is 1260037.7381022866
DP result for 181 is 1256133.8062821068


Which value of $b$ is the best now? Does it differ?


No it doesn't really differ. 81 or 91 (or even slightly higher) is still good.
The error due to bias is much greater than the error due to variance, until the
clipping parameter is really big.


## Question 3

Write an algorithm to _automatically pick the clipping parameter_ for the age column. Your algorithm should satisfy differential privacy.


In [4]:
def pick_b(epsilon):
    # Idea: try a bunch of different b values, (like question 2 prior)
    # and then find the first pair (b, b') where the noisy value decreases.
    # i.e. the noisy value for b is greater than the noisy value for b'.

    bs = range(1, 200, 20)
    # real_sum = adult["Age"].sum()

    # this forces the total privacy cost to be epsilon, without going over.
    epsilon_i = epsilon / len(bs)

    last_sum = 0

    # this satisfies len(bs) * epsilon_i for DP by sequential composition.
    for b in bs:
        clipped_sum = adult["Age"].clip(lower=0, upper=b).sum()
        dp_sum = laplace_mech(clipped_sum, sensitivity=b, epsilon=epsilon_i)
        if dp_sum < last_sum:
            return b
        else:
            last_sum = dp_sum

    # This is a safe catch to do:
    return bs[-1]  # if no decrease is found, return the last b value

    # Could also raise an exception:
    # raise Exception("No decrease found in dp_sum values.")


pick_b(1.0)

101

In [5]:
# Total privacy cost is 1.0 + 2.0 = 3.0
b = pick_b(1.0)

result = laplace_mech(
    adult["Age"].clip(lower=0, upper=b).sum(), sensitivity=b, epsilon=2.0
)

In [6]:
# TEST CASE for question 3

many_trials = np.mean([pick_b(1.0) for _ in range(100)])
assert many_trials > 70
assert many_trials < 500

In [7]:
# 09/30/2026 Notes on how to argue correctness for functions.

# Exam question that would use this:
# Make an arguement that a function or program satisfies differential privacy.

# 1. Sensitivity: I use the laplace mechanism twice. For the sum, the
#    sensitivity is 120 beacause I assume no age can be greater than 120, and
#    the sum can increase by at most 120 if I add a 1220-year-old to the data.
#    For the count, the sensitvity is 1 because all counting queries have
#    sensitvity of 1.
# 2. Composition: I use the laplace mechanism twice, the epsilons are 1.5 and
#    0.5, so the total privacy cost is 1.5 + 0.5 = 2.0 by sequential
#    composition.
# 3. Post-Processing: My return values is noisy_sum / noisy_count, both inputs
#    to this are the result of DP mechanisms, so the result of this arithmetic
#    is also DP by post-processing. (using constansts that don't depend on the
#    data are also okay.)

noisy_sum = laplace_mech(
    adult["Age"].clip(lower=0, upper=120).sum(), sensitivity=120, epsilon=1.5
)
noisy_count = laplace_mech(len(adult), sensitivity=1, epsilon=0.5)
result = noisy_sum / noisy_count

## Question 4

What is the privacy cost of your algorithm, and why?


The privacy cost of pick_b is len(bs) \* epsilon_i = epsilon
Each iteration of the loop uses epsilon_i privacy cost, and there are len(bs) iterations.


## Question 5

Write code to generate a _histogram_ of educations in the `adult` dataset.


In [8]:
def education_hist():
    # These two are the same thing:
    return adult["Education"].value_counts()
    # return adult.groupby("Education").size()


education_hist()

Education
HS-grad         10501
Some-college     7291
Bachelors        5355
Masters          1723
Assoc-voc        1382
11th             1175
Assoc-acdm       1067
10th              933
7th-8th           646
Prof-school       576
9th               514
12th              433
Doctorate         413
5th-6th           333
1st-4th           168
Preschool          51
Name: count, dtype: int64

In [9]:
# TEST CASE for question 5
h = education_hist()
assert h["HS-grad"] == 10501
assert h["12th"] == 433
assert h["Doctorate"] == 413

## Question 6

Write code to release a _differentially private histogram_ of education numbers.


In [ ]:
def dp_education_hist(epsilon):
    regular_hist = adult["Education"].value_counts()

    # Sequential composition
    epsilon_i = epsilon / len(regular_hist)
    noisy_hist = regular_hist.apply(
        lambda x: laplace_mech(x, sensitivity=1, epsilon=epsilon_i)
    )
    return noisy_hist
    # The total privacy cost is len(regular_hist) * epsilon_i = epsilon, by sequential composition.

    # Parallel composition
    # noisy_hist_parallel = regular_hist.apply(
    #     lambda x: laplace_mech(x, sensitivity=1, epsilon=epsilon)
    # )
    # return noisy_hist_parallel
    # The elements (cells) of the histogram come from disjoint subsets of the
    # dataset.
    # The total privacy cost is epsilon, by PARALLEL COMPOSITION.
    # The parallel composition version adds less noise for the same privacy
    # cost, thus giving us a better choice over sequential.


# The total privacy cost is len(regular_hist) * epsilon_i = epsilon, by sequential composition.


dp_education_hist(1.0)

Education
HS-grad         10476.863872
Some-college     7290.342504
Bachelors        5323.314454
Masters          1710.392765
Assoc-voc        1360.301672
11th             1175.851402
Assoc-acdm       1095.235173
10th              918.901303
7th-8th           589.511456
Prof-school       529.316392
9th               569.074572
12th              453.258358
Doctorate         433.860102
5th-6th           334.714919
1st-4th           144.136852
Preschool          32.334164
Name: count, dtype: float64

In [11]:
# TEST CASE for question 6
h = dp_education_hist(1.0)
assert abs(h["HS-grad"] - 10501) < 100
assert abs(h["Doctorate"] - 413) < 100

## Question 7

What is the total privacy cost of `dp_education_hist`, and why?


1. Sensitivity: sensitivity is 1 because everything is a counting query.
2. Composition: total privacy cost is len(regullar_hist) \* epsilon_i =
   epsilon, by sequential composition.
3. Post-processing: all elements of the returned histogram satisy DP.


## Question 8

Write code to generate a differentially private _contingency table_ for the Education and Sex columns of the `adult` dataset.


In [28]:
def dp_crosstab_education_sex(epsilon):
    hist = adult.groupby(["Sex", "Education"]).size()
    # Even now, when we are grouping by multiple columns, each cell of the
    # histogram is still disjoint. Therefore, we can still take advantage of
    # parallel composition here!
    # So we apply our mechanism with epsilon directly, no need to use smaller
    # epsilons for each cell.
    return hist.apply(lambda x: laplace_mech(x, sensitivity=1, epsilon=epsilon))
    # The noise added to each cell is unique.
    # The total privacy cost is just epsilon, by parallel composition.


dp_crosstab_education_sex(1.0)

Sex     Education   
Female  10th             295.397603
        11th             432.056669
        12th             144.146165
        1st-4th           45.577433
        5th-6th           85.216893
        7th-8th          160.230490
        9th              144.913661
        Assoc-acdm       421.478014
        Assoc-voc        500.728705
        Bachelors       1618.784198
        Doctorate         86.611292
        HS-grad         3388.317551
        Masters          535.852583
        Preschool         16.626359
        Prof-school       93.098458
        Some-college    2806.909816
Male    10th             639.175353
        11th             743.045109
        12th             290.149227
        1st-4th          117.634855
        5th-6th          250.420856
        7th-8th          485.346730
        9th              369.778354
        Assoc-acdm       647.981746
        Assoc-voc        882.441215
        Bachelors       3734.986424
        Doctorate        325.777386
       

In [29]:
# TEST CASE for question 8
ct = dp_crosstab_education_sex(1.0)
assert abs(ct["Female"]["10th"] - 295) < 100
assert abs(ct["Male"]["10th"] - 638) < 100
assert abs(ct["Female"]["Bachelors"] - 1619) < 100

assert abs(ct["Female"]["10th"] - 295) > 0
assert abs(ct["Male"]["10th"] - 638) > 0
assert abs(ct["Female"]["Bachelors"] - 1619) > 0

## Question 9

Does parallel composition apply for the contingency table in question 1? Why or why not?


Yes, we can. The "chunks" returned by the groupby() are disjoint, because
groupby puts rows into groups based on combinations of attribute values, and
each row has exactly one combination of attribute values.


## Question 10

Does the number of columns used in constructing the contingency table matter for privacy cost? Does it matter for accuracy?


1. For privacy cost it does not matter. You can use parallel composition for an
   n-dimensional historgram, for any value of n. This is just how groupby()
   works.
2. For utility, higher dimensional histograms have much smaller groups, and
   therefore much less signal in each histogram bin. This means you get much
   lower utility in general because the noise overwhelms the (weak) signal.
